# SMARTRENT — ไฟล์ที่ 3: การปรับปรุงโมเดลใน Phase 3 (18 ข้อมูลห้อง + 6 ข้อมูลโครงการ)

> **ลำดับสมุดโน้ตของโปรเจกต์ (3 ไฟล์ต่อเนื่องกัน):**
> 1. `1_Data_Prep.ipynb` — เตรียมข้อมูลจากไฟล์ดิบ `202306.xlsx` และคัดเลือกอัลกอริทึมหลัก (`Ridge Regression`)
> 2. `2_Phase2_Model.ipynb` — ตรวจสอบและรันโมเดลฐานของ Phase 2 (18 ตัวแปรห้อง)
> 3. **`3_Phase3_Model.ipynb`** *(ไฟล์นี้ — พระเอกของ Phase 3)* — เพิ่มข้อมูลโครงการ 6 ตัวแปร รวมเป็น **24 ตัวแปร** แล้วเทียบคะแนนกับ Phase 2

**ไฟล์นี้มีไว้ทำอะไร?**  
ต่อยอดจากคำแนะนำใน Phase 2 ที่ว่าลำพังข้อมูลในห้องยังไม่รู้ทำเลและเกรดของคอนโด ในไฟล์นี้เราจึงนำข้อมูลระดับโครงการจากแท็บ `Project` ทั้ง 6 ตัวแปร (`project_district`, `log_project_units`, `project_age_2023`, `project_max_floors`, `project_buildings`, `reported_common_amenities`) มาใช้ร่วมกับข้อมูลห้องเดิม 18 ตัวแปร เพื่อเทียบผลบน **ชุดทดสอบเดียวกัน (800 ห้อง)** แบบตรงไปตรงมา

### วิธีเปิดและกดรันทีละขั้นตอน
1. **ถ้าอ่านบนเว็บ GitHub:** เลื่อนดูโค้ดและผลลัพธ์ใต้แต่ละกล่องได้ทันทีโดยไม่ต้องกดรันใหม่
2. **ถ้ารันบนเครื่องตัวเอง (VS Code / Jupyter Notebook):**
   - ติดตั้งไลบรารีด้วยคำสั่ง `pip install -r requirements.txt`
   - กดปุ่ม **Run All** (หรือกด `Shift + Enter` ไล่ลงมาทีละกล่องจากขั้นตอนที่ 1 ถึง 5)

## ขั้นตอนที่ 1: เตรียมไลบรารีและเชื่อมต่อชุดฟังก์ชันของโปรเจกต์
กล่องโค้ดด้านล่างทำหน้าที่หาตำแหน่งโฟลเดอร์หลักของโปรเจกต์ (`ROOT`) และดึงฟังก์ชันที่เราเขียนแยกไว้ในไฟล์ `python_code/pipeline.py` เข้ามาใช้งาน

In [1]:
from pathlib import Path
import sys
import pandas as pd

# ตรวจสอบตำแหน่งโฟลเดอร์หลักของโปรเจกต์อัตโนมัติ
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
assert (ROOT / "data/train.csv").exists(), "กรุณาเปิด Notebook จากโฟลเดอร์หลัก หรือโฟลเดอร์ notebooks/"

# ดึงฟังก์ชันเตรียมข้อมูล เทรนโมเดล และวัดผลจากโฟลเดอร์ python_code/
sys.path.insert(0, str(ROOT / "python_code"))
from pipeline import load_data, fit_phase3_model, predict_phase3_model, verify_phase3_saved, metrics

## ขั้นตอนที่ 2: โหลดข้อมูล Train / Validation / Test และเช็กจำนวนห้อง
เราแบ่งข้อมูลทั้งหมด **6,425 ห้อง (จาก 230 โครงการคอนโด)** ออกเป็น 3 ชุด โดยใช้รหัสโครงการ (`project_id`) เป็นตัวแบ่งกลุ่ม เพื่อให้ห้องที่อยู่ตึกเดียวกันอยู่ในชุดเดียวกันทั้งหมด (ป้องกันข้อมูลรั่วข้ามชุด):
- **`train` (4,217 ห้อง / 138 โครงการ):** ใช้ฝึกสอนโมเดลในรอบแรก
- **`validation` (1,408 ห้อง / 46 โครงการ):** ใช้ทดลองปรับค่าและเลือกโมเดลที่ดีที่สุด
- **`train_plus_val` (5,625 ห้อง / 184 โครงการ):** นำ `train` มารวมกับ `validation` เพื่อฝึกโมเดลตัวสุดท้ายก่อนนำไปวัดผลจริง
- **`test` (800 ห้อง / 46 โครงการ):** ชุดข้อสอบจริงที่โมเดลไม่เคยเห็นมาก่อน ใช้เทียบคะแนนระหว่าง Phase 2 กับ Phase 3

In [2]:
# โหลดไฟล์ข้อมูลทั้ง 3 ชุดจากโฟลเดอร์ data/
train, val, test, dev = load_data(ROOT)

# แสดงตารางสรุปจำนวนห้อง (listings) และจำนวนโครงการคอนโด (projects) ในแต่ละชุด
summary_df = pd.DataFrame([
    {"set": name, "listings": len(df), "projects": df.project_id.nunique()}
    for name, df in [("train", train), ("validation", val), ("test", test), ("train_plus_val", dev)]
])
print(summary_df.to_string(index=False))

           set  listings  projects
         train      4217       138
    validation      1408        46
          test       800        46
train_plus_val      5625       184


## ขั้นตอนที่ 3: เปรียบเทียบผลลัพธ์ Phase 2 (18 ตัวแปร) vs Phase 3 (24 ตัวแปร)
ตารางด้านล่างแสดงผลการวัดความแม่นยำบนชุด `test` 800 ห้องเดิม:
- **`MAE_THB` (ความคลาดเคลื่อนเฉลี่ยต่อห้อง — ยิ่งต่ำยิ่งดี):** ลดลงจาก **13,866.36 บาท** เหลือ **10,013.88 บาท** (ลดลง 27.8%)
- **`RMSE_THB` (ความคลาดเคลื่อนมาตรฐานที่เน้นจับเคสทายเพี้ยนหนัก — ยิ่งต่ำยิ่งดี):** ลดลงจาก **29,879.92 บาท** เหลือ **24,660.18 บาท** (ลดลง 17.5%)
- **`R2` (สัดส่วนที่โมเดลอธิบายราคาตลาดได้ — ยิ่งใกล้ 1 ยิ่งดี):** เพิ่มขึ้นจาก **0.7089 (70.9%)** เป็น **0.8017 (80.2%)**
- **`within20_pct` (สัดส่วนห้องที่ทายคลาดเคลื่อนไม่เกิน ±20% จากราคาจริง):** เพิ่มขึ้นจาก **31.38%** เป็น **69.50%**

In [3]:
# อ่านตารางสรุปผลเปรียบเทียบ Phase 2 กับ Phase 3 จากโฟลเดอร์ results/
print(pd.read_csv(ROOT / "results/phase3_test_comparison.csv").to_string(index=False))

                                            model   n      MAE_THB     RMSE_THB       R2  within20_pct
                Phase 2: Ridge (18 Room Features) 800 13866.355268 29879.921466 0.708932        31.375
Phase 3: Ridge (18 Room + 6 Project = 24 Features) 800 10013.875842 24660.175341 0.801742        69.500


## ขั้นตอนที่ 4: โหลดโมเดลจริงมาตรวจสอบ และลองสั่งเทรนโมเดล Phase 3 ซ้ำใหม่
โค้ดกล่องนี้ทำ 2 อย่าง:
1. เรียกฟังก์ชัน `verify_phase3_saved(ROOT)` เพื่อโหลดไฟล์โมเดล `.joblib` ทั้งของ Phase 2 และ Phase 3 ที่บันทึกไว้ในโฟลเดอร์ `models/` มาทำนายชุด Test 800 ห้อง แล้วเช็กว่าได้ตัวเลขตรงกับตารางรายงานทุกทศนิยม
2. เรียกฟังก์ชัน `fit_phase3_model(dev, alpha=1.0)` เพื่อลองฝึกโมเดล Ridge Regression ของ Phase 3 ขึ้นมาใหม่สดๆ จากชุดข้อมูล `train + validation` (5,625 ห้อง) เพื่อพิสูจน์ว่าสามารถเทรนซ้ำและได้ผลลัพธ์ตรงกัน 100%

In [4]:
# 1) โหลดโมเดลที่บันทึกไว้ในโฟลเดอร์ models/ มาตรวจความถูกต้อง
print(verify_phase3_saved(ROOT).to_string(index=False))

# 2) ลองฝึกโมเดล Phase 3 (Ridge Regression, 24 ตัวแปร) ซ้ำใหม่ตั้งแต่ต้น แล้ววัดผลบนชุด Test
retrained_p3 = fit_phase3_model(dev, alpha=1.0)
retrained_pred = predict_phase3_model(retrained_p3, test)
print("Retrained Phase 3 Test metrics:", metrics(test.rental, retrained_pred, test.project_id))

                                            model      MAE_THB     RMSE_THB       R2  MAPE_pct  within20_pct  project_macro_MAE_THB
                Phase 2: Ridge (18 Room Features) 13866.355268 29879.921466 0.708932 34.100326        31.375           20341.924585
Phase 3: Ridge (18 Room + 6 Project = 24 Features) 10013.875842 24660.175341 0.801742 17.243346        69.500           20124.447570
Retrained Phase 3 Test metrics: {'MAE_THB': 10013.87584218805, 'RMSE_THB': 24660.17534085966, 'R2': 0.8017421242009261, 'MAPE_pct': 17.243346, 'within20_pct': 69.5, 'project_macro_MAE_THB': 20124.447570}


## ขั้นตอนที่ 5: ทดลองทำนายราคาค่าเช่าจริง (ตัวอย่าง 5 ห้องแรกจากชุด Test)
ลองนำข้อมูล 5 ห้องแรกในชุด `test` มาให้โมเดล Phase 3 ทำนายค่าเช่า (`predicted_phase3`) เพื่อเทียบกับค่าเช่าจริงที่เจ้าของประกาศ (`rental`)

In [5]:
# เลือก 5 ห้องแรกจากชุด Test มาทำนายและแสดงผลเทียบกับค่าเช่าจริง
sample = test.head(5).copy()
sample["predicted_phase3"] = predict_phase3_model(retrained_p3, sample).round(0)
print(sample[["room_id", "project_name", "project_district", "size_sqm", "rental", "predicted_phase3"]].to_string(index=False))

 room_id                   project_name      project_district  size_sqm  rental  predicted_phase3
       1        Beat Bangwa Interchange Phasi Charoen Bangkok     27.00   10000            9077.0
      23        Beat Bangwa Interchange Phasi Charoen Bangkok     29.15    9500            9907.0
      24 The Tempo Grand Sathorn - Wutthakat    Chom Thong Bangkok     29.00   12000           10623.0
      47 Aspire Sathorn - Taksin (Timber Zone)     Thon Buri Bangkok     45.00   15000           16302.0
      54 The Tempo Grand Sathorn - Wutthakat    Chom Thong Bangkok     35.00   12000           11952.0
